# Credit Card Fraud Detection: Extreme Imbalance & Cost Optimization
**Category 03: Machine Learning | Project 04**

This walkthrough implements a production-grade machine learning system for credit card fraud detection under severe class imbalance (~0.8% fraud).

### Key Highlights:
- **The Accuracy Paradox**: Why a naive baseline predicting zero fraud achieves 99.2% accuracy but catches 0% of fraud.
- **Outlier-Resistant Scaling**: Using `RobustScaler` on heavy-tailed transaction amounts and timestamps.
- **Data Leakage Prevention**: Resampling techniques applied strictly to the training fold.
- **Candidate Model Regimes**: Benchmarking baseline unweighted, cost-sensitive balanced, and undersampled algorithms across Logistic Regression, Decision Tree, and Random Forest.
- **Evaluation Specialized for Imbalance**: Comparing **PR-AUC (Average Precision)**, **Recall**, and **ROC-AUC**.
- **Asymmetric Financial Loss Matrix**: Formulating operational trade-offs between False Positives ($10 investigation cost) and False Negatives ($400 unrecovered fraud loss).
- **Real-Time Risk Decisioning**: Automated authorization tiers (Approve, Challenge with 2FA, Decline).

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import load_fraud_data
from src.data_preprocessing import preprocess_fraud_data, prepare_stratified_train_test, resample_training_data
from src.model_training import train_fraud_classifiers, save_best_fraud_model
from src.model_evaluation import evaluate_fraud_models, plot_fraud_evaluations, calculate_business_cost_impact
from src.predictor import FraudPredictor

print("Environment initialized successfully.")

## 1. Data Ingestion & Class Imbalance Exploration
We load the transaction dataset and inspect the extreme class imbalance.

In [ ]:
raw_df = load_fraud_data()
print(f"Raw dataset shape: {raw_df.shape}")
fraud_rate = raw_df['Class'].mean()
print(f"Fraudulent Transactions: {raw_df['Class'].sum():,} ({fraud_rate:.3%})")
print(f"Legitimate Transactions: {(raw_df['Class'] == 0).sum():,} ({1 - fraud_rate:.3%})")
raw_df.head(5)

## 2. Preprocessing & Outlier-Resistant Scaling
Transaction amounts and timestamps contain extreme outliers; we apply `RobustScaler` (median & IQR).

In [ ]:
clean_df, report = preprocess_fraud_data(raw_df)
print("Preprocessing Diagnostic Report:", report)
clean_df[['Time', 'Amount', 'V4', 'V10', 'V12', 'V14', 'Class']].describe()

## 3. Stratified Train-Test Splitting (Preventing Leakage)
Class stratification ensures that the held-out test set precisely mirrors the true minority prevalence.

In [ ]:
X_train, X_test, y_train, y_test = prepare_stratified_train_test(clean_df, test_size=0.20, random_state=42)
print(f"Train samples: {len(X_train)} (Fraud: {y_train.sum()})")
print(f"Test samples:  {len(X_test)} (Fraud: {y_test.sum()})")

## 4. Model Training Across Balancing Regimes
We train five pipeline configurations:
1. **Baseline Logistic Regression**: Unweighted
2. **Weighted Logistic Regression**: Cost-sensitive balanced weights
3. **Undersampled Logistic Regression**: Resampled majority class on train set only
4. **Balanced Decision Tree**: Depth-constrained tree
5. **Balanced Random Forest**: Bagged ensemble with balanced weights

In [ ]:
models = train_fraud_classifiers(X_train, y_train, random_state=42)
for name in models:
    print(f"[+] Trained pipeline: {name}")

## 5. Benchmarking Beyond Accuracy (PR-AUC & Business Loss)
Benchmarking all models on the held-out test set.

In [ ]:
summary_df, predictions, probabilities = evaluate_fraud_models(models, X_test, y_test)
summary_df

## 6. Asymmetric Loss Economics ($10 FP vs $400 FN)
In banking operations, False Negatives carry direct financial loss from fraudulent chargebacks, whereas False Positives only cause mild cardholder friction.

In [ ]:
for name, y_pred in predictions.items():
    loss_info = calculate_business_cost_impact(y_test.to_numpy(), y_pred)
    print(f"{name:<34} | FPs: {loss_info['false_positives']:<2} | FNs: {loss_info['false_negatives']:<2} | Total Loss: ${loss_info['total_financial_loss']:,.0f}")

## 7. Model Persistence & Real-Time Decisioning
Serializing top pipeline and verifying production inference on transaction payloads.

In [ ]:
best_name = summary_df.iloc[0]['Model']
best_pipe = models[best_name]
best_metrics = summary_df.iloc[0].to_dict()
saved_path = save_best_fraud_model(best_pipe, best_name, best_metrics)
print(f"Serialized model to: {saved_path}")

predictor = FraudPredictor(model_path=saved_path)
test_txn = {
    'Time': 75000.0,
    'Amount': 1850.00,
    'V4': 4.5,
    'V10': -4.5,
    'V12': -5.1,
    'V14': -6.8,
    'V17': -4.2,
}
result = predictor.score_transaction(test_txn)
for k, v in result.items():
    print(f"{k}: {v}")